# Конфиг и проверка ГПУ

In [1]:
from pathlib import Path
import json
import torch
import os

# Вся HF-экосистема (hub, datasets и т.д.) — на диск S:
os.environ["HF_HOME"] = r"S:\hw\aspa\.cache\huggingface"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ROOT = Path(r"S:\hw\aspa")
DATA_PATH = ROOT / "llm_fc_hypernym_prediction/output/dataset/validate_10_old.jsonl"
MODEL_ID = "Qwen/Qwen3.5-2B"
OUTPUT_DIR = ROOT / "output/lora_smoke"

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0))

torch: 2.11.0+cu128
cuda: True NVIDIA GeForce RTX 5080


# Загрузить и посмотреть данные

In [2]:
records = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print("records:", len(records))
print("roles:", [m["role"] for m in records[0]["messages"]])
print("final:", records[0]["messages"][-1]["content"][:120])

records: 20
roles: ['system', 'user', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant']
final: hyponym of 147309-N (неучастие, отказ от участия)


# Загрузка туллзов

In [3]:
import sys
sys.path.insert(0, str(ROOT / "llm_fc_hypernym_prediction"))
from utils.tools import hyponym_only

TOOLS = hyponym_only
TOOLS[0]  # посмотреть схему get_hyponyms

{'type': 'function',
 'function': {'name': 'get_hyponyms',
  'description': 'Navigate the RuWordNet taxonomy by retrieving hyponyms (more specific concepts) of a given synset. Returns formatted markdown with: synset name, ID, associated words, and list of child hyponyms. When node_id is null, returns all root nodes (top-level concepts). Use this to explore the taxonomy tree level by level.',
  'parameters': {'type': 'object',
   'properties': {'node_id': {'type': ['string', 'null'],
     'description': "The synset ID to get hyponyms for. Use 'null' or null to retrieve all root nodes (top-level concepts in the taxonomy). Use specific synset ID like '123456-N' to get its children."}},
   'required': ['node_id']}}}

# Загрузка tokenizer + проверка chat template

In [4]:
import copy
import json

def normalize_messages_for_template(messages):
    """Qwen3.5 chat template ждёт arguments как dict, не JSON-строку."""
    msgs = copy.deepcopy(messages)
    for msg in msgs:
        if msg.get("role") != "assistant":
            continue
        for tc in msg.get("tool_calls") or []:
            fn = tc.get("function", tc)
            args = fn.get("arguments")
            if isinstance(args, str):
                fn["arguments"] = json.loads(args)
    return msgs

In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
sample = normalize_messages_for_template(records[0]["messages"][:3])
text = tokenizer.apply_chat_template(
    sample,
    tools=TOOLS,
    tokenize=False,
    add_generation_prompt=False,
    enable_thinking=False,   # см. ниже — не chat_template_kwargs
)
print(text[:2000])

<|im_start|>system
# Tools

You have access to the following functions:

<tools>
{"type": "function", "function": {"name": "get_hyponyms", "description": "Navigate the RuWordNet taxonomy by retrieving hyponyms (more specific concepts) of a given synset. Returns formatted markdown with: synset name, ID, associated words, and list of child hyponyms. When node_id is null, returns all root nodes (top-level concepts). Use this to explore the taxonomy tree level by level.", "parameters": {"type": "object", "properties": {"node_id": {"type": ["string", "null"], "description": "The synset ID to get hyponyms for. Use 'null' or null to retrieve all root nodes (top-level concepts in the taxonomy). Use specific synset ID like '123456-N' to get its children."}}, "required": ["node_id"]}}}
</tools>

If you choose to call a function ONLY reply in the following format with NO suffix:

<tool_call>
<function=example_function_name>
<parameter=example_parameter_1>
value_1
</parameter>
<parameter=example_p

# Загрузка модели

### Загрузка 16-битного Qwen3.5-4B

In [6]:
from transformers import AutoModelForCausalLM
import gc

gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False  # для обучения

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


[ERROR] `loss` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in s:\hw\aspa\.ml_venv\Lib\site-packages\transformers\models\qwen3_5\modeling_qwen3_5.py.
[ERROR] `logits` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in s:\hw\aspa\.ml_venv\Lib\site-packages\transformers\models\qwen3_5\modeling_qwen3_5.py.


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
s:\hw\aspa\.ml_venv\Lib\site-packages\accelerate\utils\modeling.py:817: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  _ = torch.tensor([0], device=i)


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

# LoRA + SFT

### Подготовка датасета

In [7]:
from datasets import Dataset

records_for_train = [
    {
        "messages": normalize_messages_for_template(r["messages"]),
        "tools": TOOLS,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    for r in records   # все 20, не [:2]
]
ds = Dataset.from_list(records_for_train)

### Проверка длин до обучения

In [8]:
lengths = []
for ex in records_for_train:
    result = tokenizer.apply_chat_template(
        ex["messages"],
        tools=ex["tools"],
        tokenize=True,
        return_dict=True,
        enable_thinking= False
    )
    n = len(result["input_ids"])
    # print(result.keys())
    print(n)

# print(f"examples: {len(lengths)}, min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)/len(lengths):.0f}")

14362
13473
14361
12158
13108
11949
21334
20864
20331
15479
4223
12335
19973
20022
7627
10446
6446
8086
8570
12530


### Запуск для 16-битного Qwen3.5-4B

In [ ]:
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer
import gc

# 3. LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 4. Обучение — БЕЗ formatting_func
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=1,
    save_steps=50,
    max_length=4000, 
    # max_length=8192, при этом значении падает по памяти
    assistant_only_loss=True,
    gradient_checkpointing=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=ds,
    processing_class=tokenizer,
    # formatting_func НЕ передаём
)

gc.collect()
torch.cuda.empty_cache()

trainer.train()
trainer.save_model(str(OUTPUT_DIR))